In [1]:
!find /kaggle/input -maxdepth 4 -type d | head -100

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/sujeethkasukurthi
/kaggle/input/datasets/sujeethkasukurthi/abcdef
/kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload
/kaggle/input/datasets/quanglvitlm
/kaggle/input/datasets/quanglvitlm/musdb18-hq
/kaggle/input/datasets/quanglvitlm/musdb18-hq/valid
/kaggle/input/datasets/quanglvitlm/musdb18-hq/test
/kaggle/input/datasets/quanglvitlm/musdb18-hq/train


In [2]:
from pathlib import Path

src = next(Path("/kaggle/input").rglob("kaggle_rope_unet_phases_upload"))
print("PROJECT SOURCE:", src)

!cp -r "{src}" /kaggle/working/project

PROJECT = "/kaggle/working/project"
MUSDB = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"

print("PROJECT =", PROJECT)
print("MUSDB =", MUSDB)

PROJECT SOURCE: /kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload
PROJECT = /kaggle/working/project
MUSDB = /kaggle/input/datasets/quanglvitlm/musdb18-hq


In [3]:
from pathlib import Path

src = next(Path("/kaggle/input").rglob("kaggle_rope_unet_phases_upload"))
print(src)

/kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload


In [4]:
!cp -r "{src}" /kaggle/working/project

PROJECT = "/kaggle/working/project"
MUSDB = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"

In [5]:
!python "$PROJECT/scripts/count_baby_params.py"
!python "$PROJECT/scripts/verify_musdb_hq_layout.py" --dataset-root "$MUSDB"

student hidden size: 384
student trainable parameters: 5,762,304
student weights file: /kaggle/working/project/student_best_audio_weights_only.pt
student weights size MB: 22.04
train: 100 songs; official expected 100
  all song folders have mixture/bass/drums/other/vocal(s)
  checked mixtures are stereo 44.1 kHz
test: 50 songs; official expected 50
  all song folders have mixture/bass/drums/other/vocal(s)
  checked mixtures are stereo 44.1 kHz


In [ ]:
!python "$PROJECT/scripts/train_audio_phases.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --run-dir /kaggle/working/runs/phase1_v1_audio \
  --model-variant v1 \
  --init-checkpoint "$PROJECT/student_best_audio_weights_only.pt" \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 30 \
  --steps-per-epoch 300 \
  --val-steps 50 \
  --batch-size 1 \
  --lr 5e-5 \
  --wave-weight 1.0 \
  --stft-weight 0.2 \
  --hidden-weight 0.05 \
  --teacher-stem-weight 0.0 \
  --mixture-weight 0.05 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1285.16it/s, Materializing para
loaded 176/176 matching tensors from /kaggle/working/project/student_best_audio_weights_only.pt
variant: v1
student params: 5,762,304
train songs: 90 | val songs: 10
run dir: /kaggle/working/runs/phase1_v1_audio
val: 100%|██████████████████████████████████████| 50/50 [00:13<00:00,  3.81it/s]
epoch 1: train 0.400926 | val 0.024545
new best: 0.024545
val: 100%|██████████████████████████████████████| 50/50 [00:11<00:00,  4.26it/s]
epoch 2: train 0.398621 | val 0.026747
val: 100%|██████████████████████████████████████| 50/50 [00:10<00:00,  4.70it/s]
epoch 3: train 0.393097 | val 0.028750
val: 100%|██████████████████████████████████████| 50/50 [00:10<00:00,  4.81it/s]
epoch 4: train 0.394351 | val 0.027293
val: 100%|██████████████████████████████████████| 50/50 [00:10<00:00,  4.86it/s]
epoch 5: train 0.393151 | val 0.025481
val: 100%|████████████████████████████

In [7]:
!find /kaggle/working/runs/phase1_v1_audio -maxdepth 1 -type f -name "*.pt" -ls
!cat /kaggle/working/runs/phase1_v1_audio/history.json

  1048654  67740 -rw-r--r--   1 root     root     69363217 May 30 14:21 /kaggle/working/runs/phase1_v1_audio/best_audio.pt
  1048653  67740 -rw-r--r--   1 root     root     69363217 May 30 15:13 /kaggle/working/runs/phase1_v1_audio/last_audio.pt
[
  {
    "epoch": 1,
    "train_loss": 0.4009263129035632,
    "val_loss": 0.024545361381678957
  },
  {
    "epoch": 2,
    "train_loss": 0.3986209842562676,
    "val_loss": 0.02674710450362909
  },
  {
    "epoch": 3,
    "train_loss": 0.3930966958403587,
    "val_loss": 0.02874960212218866
  },
  {
    "epoch": 4,
    "train_loss": 0.3943509547909101,
    "val_loss": 0.027292549579874502
  },
  {
    "epoch": 5,
    "train_loss": 0.39315126846234005,
    "val_loss": 0.025480705497284362
  },
  {
    "epoch": 6,
    "train_loss": 0.39031963715950646,
    "val_loss": 0.023755977886098664
  },
  {
    "epoch": 7,
    "train_loss": 0.39310598934690155,
    "val_loss": 0.026313526208723487
  },
  {
    "epoch": 8,
    "train_loss": 0.39070732235

In [8]:
!python "$PROJECT/scripts/eval_audio_student_sdr.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase1_v1_audio/best_audio.pt \
  --model-variant v1 \
  --out-dir /kaggle/working/sdr_phase1_v1 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1272.98it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [05:50<00:00,  7.02s/it]
SUMMARY
bass: 3.618 dB
drums: 5.380 dB
other: 3.017 dB
vocal: 4.484 dB
mean: 4.125 dB
saved: /kaggle/working/sdr_phase1_v1/student_sdr_results.json


In [9]:
!mkdir -p /kaggle/working/final_save
!cp -r /kaggle/working/runs /kaggle/working/final_save/runs
!cp -r /kaggle/working/sdr_phase1_v1 /kaggle/working/final_save/sdr_phase1_v1
!cp -r "$PROJECT" /kaggle/working/final_save/project_code
!cd /kaggle/working && zip -r final_phase1_results.zip final_save

  adding: final_save/ (stored 0%)
  adding: final_save/project_code/ (stored 0%)
  adding: final_save/project_code/best_audio_training_checkpoint.pt (deflated 8%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/ (stored 0%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/best_audio_training_checkpoint.pt (deflated 8%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/KAGGLE_TRAINING.md (deflated 63%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/ (stored 0%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/configuration_bs_roformer.py (deflated 71%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/modeling_bs_roformer.py (deflated 76%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/model.safetensors (deflated 24%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/config.json (deflated 7

In [11]:
!python "$PROJECT/scripts/train_audio_phases.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --run-dir /kaggle/working/runs/phase2_v1_teacher_audio \
  --model-variant v1 \
  --init-checkpoint /kaggle/working/runs/phase1_v1_audio/best_audio.pt \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 30 \
  --steps-per-epoch 200 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 3e-5 \
  --wave-weight 1.0 \
  --stft-weight 0.2 \
  --hidden-weight 0.05 \
  --teacher-stem-weight 0.3 \
  --mixture-weight 0.05 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1276.04it/s, Materializing para
loaded 176/176 matching tensors from /kaggle/working/runs/phase1_v1_audio/best_audio.pt
variant: v1
student params: 5,762,304
train songs: 90 | val songs: 10
run dir: /kaggle/working/runs/phase2_v1_teacher_audio
val: 100%|██████████████████████████████████████| 30/30 [00:04<00:00,  6.43it/s]
epoch 1: train 0.389487 | val 0.025663
new best: 0.025663
val: 100%|██████████████████████████████████████| 30/30 [00:04<00:00,  6.78it/s]
epoch 2: train 0.388559 | val 0.026835
val: 100%|██████████████████████████████████████| 30/30 [00:04<00:00,  7.03it/s]
epoch 3: train 0.392245 | val 0.025139
new best: 0.025139
val: 100%|██████████████████████████████████████| 30/30 [00:04<00:00,  6.52it/s]
epoch 4: train 0.386337 | val 0.025221
val: 100%|██████████████████████████████████████| 30/30 [00:04<00:00,  6.52it/s]
epoch 5: train 0.388268 | val 0.024967
new best: 0.024967


In [12]:
!python "$PROJECT/scripts/eval_audio_student_sdr.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase2_v1_teacher_audio/best_audio.pt \
  --model-variant v1 \
  --out-dir /kaggle/working/sdr_phase2_v1 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1274.17it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [05:42<00:00,  6.84s/it]
SUMMARY
bass: 3.499 dB
drums: 5.437 dB
other: 2.997 dB
vocal: 4.599 dB
mean: 4.133 dB
saved: /kaggle/working/sdr_phase2_v1/student_sdr_results.json


In [13]:
!python "$PROJECT/scripts/train_audio_phases.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --run-dir /kaggle/working/runs/phase4_v2_axial \
  --model-variant v2 \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 12 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 8e-5 \
  --remix-prob 0.5 \
  --wave-weight 1.0 \
  --stft-weight 0.25 \
  --hidden-weight 0.05 \
  --teacher-stem-weight 0.2 \
  --mixture-weight 0.05 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1236.32it/s, Materializing para
variant: v2
student params: 20,500,826
train songs: 90 | val songs: 10
run dir: /kaggle/working/runs/phase4_v2_axial
val: 100%|██████████████████████████████████████| 30/30 [00:10<00:00,  2.79it/s]
epoch 1: train 0.674564 | val 0.031132
new best: 0.031132
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.26it/s]
epoch 2: train 0.628378 | val 0.038013
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.37it/s]
epoch 3: train 0.617605 | val 0.033436
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.53it/s]
epoch 4: train 0.608702 | val 0.035884
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.22it/s]
epoch 5: train 0.603193 | val 0.031562
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.60it/s]
epoch 6: train 0.593218 | val 0.030301
new best: 0.0

In [14]:
!python "$PROJECT/scripts/eval_audio_student_sdr.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase4_v2_axial/best_audio.pt \
  --model-variant v2 \
  --out-dir /kaggle/working/sdr_phase4_v2 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1220.31it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [04:53<00:00,  5.87s/it]
SUMMARY
bass: 2.367 dB
drums: 2.997 dB
other: 1.304 dB
vocal: 1.034 dB
mean: 1.926 dB
saved: /kaggle/working/sdr_phase4_v2/student_sdr_results.json


In [15]:
!python "$PROJECT/scripts/train_audio_phases.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --run-dir /kaggle/working/runs/phase4a_v2_hidden_warmup \
  --model-variant v2 \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 25 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 1e-4 \
  --remix-prob 0.0 \
  --wave-weight 0.0 \
  --stft-weight 0.0 \
  --hidden-weight 1.0 \
  --teacher-stem-weight 0.0 \
  --mixture-weight 0.0 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1260.02it/s, Materializing para
variant: v2
student params: 20,500,826
train songs: 90 | val songs: 10
run dir: /kaggle/working/runs/phase4a_v2_hidden_warmup
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.96it/s]
epoch 1: train 11.756431 | val 0.035522
new best: 0.035522
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.34it/s]
epoch 2: train 10.959250 | val 0.038955
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.46it/s]
epoch 3: train 10.706429 | val 0.035551
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.41it/s]
epoch 4: train 10.477521 | val 0.037909
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.54it/s]
epoch 5: train 10.289901 | val 0.034803
new best: 0.034803
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.50it/s]
epoch 6: train 10.4

In [16]:
!python "$PROJECT/scripts/eval_audio_student_sdr.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase4a_v2_hidden_warmup/best_audio.pt \
  --model-variant v2 \
  --out-dir /kaggle/working/sdr_phase4a_v2_hidden_warmup \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1245.30it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [05:49<00:00,  6.99s/it]
SUMMARY
bass: 2.548 dB
drums: 3.486 dB
other: 1.415 dB
vocal: 1.293 dB
mean: 2.185 dB
saved: /kaggle/working/sdr_phase4a_v2_hidden_warmup/student_sdr_results.json


In [17]:
!python "$PROJECT/scripts/eval_audio_student_sdr.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase4a_v2_hidden_warmup/last_audio.pt \
  --model-variant v2 \
  --out-dir /kaggle/working/sdr_phase4a_v2_hidden_warmup_last \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1273.42it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [04:48<00:00,  5.76s/it]
SUMMARY
bass: 2.548 dB
drums: 3.486 dB
other: 1.415 dB
vocal: 1.293 dB
mean: 2.185 dB
saved: /kaggle/working/sdr_phase4a_v2_hidden_warmup_last/student_sdr_results.json


In [18]:
!python "$PROJECT/scripts/train_audio_phases.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --run-dir /kaggle/working/runs/phase4b_v2_audio_finetune \
  --model-variant v2 \
  --init-checkpoint /kaggle/working/runs/phase4a_v2_hidden_warmup/last_audio.pt \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 20 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 3e-5 \
  --remix-prob 0.0 \
  --wave-weight 1.0 \
  --stft-weight 0.2 \
  --hidden-weight 0.1 \
  --teacher-stem-weight 0.3 \
  --mixture-weight 0.05 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1232.26it/s, Materializing para
loaded 292/292 matching tensors from /kaggle/working/runs/phase4a_v2_hidden_warmup/last_audio.pt
variant: v2
student params: 20,500,826
train songs: 90 | val songs: 10
run dir: /kaggle/working/runs/phase4b_v2_audio_finetune
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.49it/s]
epoch 1: train 0.959633 | val 0.028581
new best: 0.028581
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.40it/s]
epoch 2: train 0.951034 | val 0.031452
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.61it/s]
epoch 3: train 0.946790 | val 0.029203
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.41it/s]
epoch 4: train 0.949627 | val 0.031229
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.62it/s]
epoch 5: train 0.950662 | val 0.028955
val: 100%|████████████████

In [19]:
!python "$PROJECT/scripts/eval_audio_student_sdr.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase4b_v2_audio_finetune/best_audio.pt \
  --model-variant v2 \
  --out-dir /kaggle/working/sdr_phase4b_v2_audio_finetune \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1252.26it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [05:11<00:00,  6.22s/it]
SUMMARY
bass: 2.825 dB
drums: 3.848 dB
other: 2.238 dB
vocal: 2.478 dB
mean: 2.847 dB
saved: /kaggle/working/sdr_phase4b_v2_audio_finetune/student_sdr_results.json


In [20]:
!python "$PROJECT/scripts/train_audio_phases.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --run-dir /kaggle/working/runs/phase3_v1_remix \
  --model-variant v1 \
  --init-checkpoint /kaggle/working/runs/phase2_v1_teacher_audio/best_audio.pt \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 20 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 2e-5 \
  --remix-prob 0.5 \
  --wave-weight 1.0 \
  --stft-weight 0.2 \
  --hidden-weight 0.05 \
  --teacher-stem-weight 0.2 \
  --mixture-weight 0.05 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1223.01it/s, Materializing para
loaded 176/176 matching tensors from /kaggle/working/runs/phase2_v1_teacher_audio/best_audio.pt
variant: v1
student params: 5,762,304
train songs: 90 | val songs: 10
run dir: /kaggle/working/runs/phase3_v1_remix
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.65it/s]
epoch 1: train 0.412240 | val 0.021857
new best: 0.021857
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.97it/s]
epoch 2: train 0.407046 | val 0.030472
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.92it/s]
epoch 3: train 0.407489 | val 0.027537
val: 100%|██████████████████████████████████████| 30/30 [00:04<00:00,  6.10it/s]
epoch 4: train 0.405774 | val 0.027088
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.52it/s]
epoch 5: train 0.408573 | val 0.021829
new best: 0.021829
val: 100%|█████████

In [21]:
!python "$PROJECT/scripts/eval_audio_student_sdr.py" \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase3_v1_remix/best_audio.pt \
  --model-variant v1 \
  --out-dir /kaggle/working/sdr_phase3_v1_remix \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1243.30it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [05:16<00:00,  6.32s/it]
SUMMARY
bass: 3.501 dB
drums: 5.419 dB
other: 3.176 dB
vocal: 4.949 dB
mean: 4.261 dB
saved: /kaggle/working/sdr_phase3_v1_remix/student_sdr_results.json


In [22]:
!mkdir -p /kaggle/working/final_save

# checkpoints
!cp -r /kaggle/working/runs /kaggle/working/final_save/runs

# SDR results
!cp -r /kaggle/working/sdr_phase1_v1 /kaggle/working/final_save/sdr_phase1_v1 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase2_v1 /kaggle/working/final_save/sdr_phase2_v1 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase3_v1_remix /kaggle/working/final_save/sdr_phase3_v1_remix 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase3_v1_remix_last /kaggle/working/final_save/sdr_phase3_v1_remix_last 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase4_v2 /kaggle/working/final_save/sdr_phase4_v2 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase4a_v2_hidden_warmup /kaggle/working/final_save/sdr_phase4a_v2_hidden_warmup 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase4b_v2_audio_finetune /kaggle/working/final_save/sdr_phase4b_v2_audio_finetune 2>/dev/null || true

# project code
!cp -r "$PROJECT" /kaggle/working/final_save/project_code

# make one zip
!cd /kaggle/working && zip -r final_all_results.zip final_save

  adding: final_save/ (stored 0%)
  adding: final_save/sdr_phase4b_v2_audio_finetune/ (stored 0%)
  adding: final_save/sdr_phase4b_v2_audio_finetune/student_sdr_results.json (deflated 64%)
  adding: final_save/project_code/ (stored 0%)
  adding: final_save/project_code/best_audio_training_checkpoint.pt (deflated 8%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/ (stored 0%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/best_audio_training_checkpoint.pt (deflated 8%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/KAGGLE_TRAINING.md (deflated 63%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/ (stored 0%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/configuration_bs_roformer.py (deflated 71%)
  adding: final_save/project_code/kaggle_rope_unet_phases_upload/teacher_model/modeling_bs_roformer.py (deflated 76%)
  adding: final_save/project_code/kaggle_rope_unet_phases_u

In [25]:
# Patch teacher eval script to force safe float32 STFT windows
from pathlib import Path

p = Path("/kaggle/working/eval_teacher_exact_sdr.py")
text = p.read_text()

old = """    model = AutoModel.from_pretrained(
        str(Path(args.project) / "teacher_model"),
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(args.device).float().eval()

    for p in model.parameters():
        p.requires_grad_(False)
"""

new = """    model = AutoModel.from_pretrained(
        str(Path(args.project) / "teacher_model"),
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(args.device).float().eval()

    # Important Kaggle/Torch safety fix:
    # Replace loaded STFT/iSTFT windows with fresh finite float32 Hann windows.
    model.stft_window = torch.hann_window(
        model.stft_kwargs.get("win_length") or model.stft_kwargs["n_fft"],
        periodic=True,
        device=args.device,
        dtype=torch.float32,
    )
    model.stft_out_window = torch.hann_window(
        model.stft_out_kwargs.get("win_length") or model.stft_out_kwargs["n_fft"],
        periodic=True,
        device=args.device,
        dtype=torch.float32,
    )

    print("stft_window finite:", torch.isfinite(model.stft_window).all().item())
    print("stft_out_window finite:", torch.isfinite(model.stft_out_window).all().item())

    for p in model.parameters():
        p.requires_grad_(False)
"""

p.write_text(text.replace(old, new))
print("patched teacher eval script")

patched teacher eval script


In [27]:
%%writefile /kaggle/working/eval_student_overlap_sdr.py
import argparse
import json
import sys
from pathlib import Path

import soundfile as sf
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModel

sys.path.insert(0, "/kaggle/working/project")

from student.rope_unet import RopeReplacementUNet
from student.axial_rope_unet_v2 import AxialRopeUNetV2
from student.teacher_features import waveform_to_teacher_features, student_features_to_stems

STEMS = ("bass", "drums", "other", "vocal")


def torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def read_audio(path, sample_rate):
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    x = torch.from_numpy(audio).transpose(0, 1).contiguous()
    if x.shape[0] == 1:
        x = x.repeat(2, 1)
    elif x.shape[0] > 2:
        x = x[:2]
    if sr != sample_rate:
        new_len = round(x.shape[-1] * sample_rate / sr)
        x = F.interpolate(x.unsqueeze(0), size=new_len, mode="linear", align_corners=False).squeeze(0)
    return x


def find_audio(song_dir, names):
    for name in names:
        for ext in (".wav", ".flac"):
            p = song_dir / f"{name}{ext}"
            if p.exists():
                return p
    raise FileNotFoundError(f"Missing {names} in {song_dir}")


def find_songs(root):
    root = Path(root)
    if (root / "mixture.wav").exists() or (root / "mixture.flac").exists():
        return [root]
    return sorted(
        p for p in root.rglob("*")
        if p.is_dir() and ((p / "mixture.wav").exists() or (p / "mixture.flac").exists())
    )


def make_student(variant, hidden_size):
    if variant == "v1":
        return RopeReplacementUNet(hidden_size=hidden_size)
    if variant == "v2":
        return AxialRopeUNetV2(hidden_size=hidden_size)
    raise ValueError(variant)


def simple_sdr(pred, target):
    n = min(pred.shape[-1], target.shape[-1])
    pred = pred[..., :n].float()
    target = target[..., :n].float()
    return float(10.0 * torch.log10((target.pow(2).sum() + 1e-8) / ((target - pred).pow(2).sum() + 1e-8)))


@torch.no_grad()
def separate_overlap(teacher, student, mixture, chunk_size, hop_size, device):
    total_len = mixture.shape[-1]
    output = torch.zeros(4, 2, total_len, dtype=torch.float32)
    weight_sum = torch.zeros(total_len, dtype=torch.float32)

    # Raised-cosine style weight, with nonzero edges.
    weight = torch.hann_window(chunk_size, periodic=False).float()
    weight = weight.clamp_min(0.05)

    starts = list(range(0, max(1, total_len - chunk_size + 1), hop_size))
    if not starts or starts[-1] + chunk_size < total_len:
        starts.append(max(0, total_len - chunk_size))

    for start in starts:
        end = min(start + chunk_size, total_len)
        chunk = mixture[:, start:end]
        real_len = chunk.shape[-1]
        if real_len < chunk_size:
            chunk = F.pad(chunk, (0, chunk_size - real_len))

        raw = chunk.unsqueeze(0).to(device)
        x = waveform_to_teacher_features(teacher, raw)
        h = student(x)
        pred = student_features_to_stems(teacher, h, chunk_size, raw).cpu()[0, :, :, :real_len]

        w = weight[:real_len]
        output[:, :, start:end] += pred * w.view(1, 1, -1)
        weight_sum[start:end] += w

    output = output / weight_sum.clamp_min(1e-8).view(1, 1, -1)
    return output


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--project", required=True)
    parser.add_argument("--dataset-root", required=True)
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--model-variant", choices=("v1", "v2"), default="v1")
    parser.add_argument("--out-dir", default="/kaggle/working/sdr_overlap")
    parser.add_argument("--device", default="cuda")
    parser.add_argument("--chunk-seconds", type=float, default=8.0)
    parser.add_argument("--overlap", type=float, default=0.5)
    parser.add_argument("--max-songs", type=int, default=0)
    args = parser.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    teacher = AutoModel.from_pretrained(
        str(Path(args.project) / "teacher_model"),
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(args.device).float().eval()

    for p in teacher.parameters():
        p.requires_grad_(False)

    ckpt = torch_load(args.checkpoint, args.device)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    student = make_student(args.model_variant, int(teacher.config.hidden_size)).to(args.device).eval()
    student.load_state_dict(state)

    sr = int(teacher.config.wave_sample_rate)
    chunk_size = int(args.chunk_seconds * sr)
    hop_size = int(chunk_size * (1.0 - args.overlap))

    songs = find_songs(args.dataset_root)
    if args.max_songs:
        songs = songs[:args.max_songs]

    totals = {s: [] for s in STEMS}
    rows = []

    for song_dir in tqdm(songs, desc="songs"):
        mix = read_audio(find_audio(song_dir, ("mixture",)), sr)
        targets = {
            "bass": read_audio(find_audio(song_dir, ("bass", "target_bass")), sr),
            "drums": read_audio(find_audio(song_dir, ("drums", "target_drums")), sr),
            "other": read_audio(find_audio(song_dir, ("other", "target_other")), sr),
            "vocal": read_audio(find_audio(song_dir, ("vocals", "vocal", "target_vocals", "target_vocal")), sr),
        }

        pred = separate_overlap(teacher, student, mix, chunk_size, hop_size, args.device)

        row = {"song": song_dir.name}
        for i, stem in enumerate(STEMS):
            row[stem] = simple_sdr(pred[i], targets[stem])
            totals[stem].append(row[stem])
        row["mean"] = sum(row[s] for s in STEMS) / 4
        rows.append(row)

        print(
            f"{song_dir.name}: bass {row['bass']:.3f} | drums {row['drums']:.3f} | "
            f"other {row['other']:.3f} | vocal {row['vocal']:.3f} | mean {row['mean']:.3f}"
        )

    summary = {stem: sum(vals) / max(1, len(vals)) for stem, vals in totals.items()}
    summary["mean"] = sum(summary.values()) / 4

    print("SUMMARY")
    for stem in STEMS:
        print(f"{stem}: {summary[stem]:.3f} dB")
    print(f"mean: {summary['mean']:.3f} dB")

    out_path = out_dir / "student_overlap_sdr_results.json"
    out_path.write_text(json.dumps({"summary": summary, "songs": rows}, indent=2))
    print(f"saved: {out_path}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/eval_student_overlap_sdr.py


In [28]:
!python /kaggle/working/eval_student_overlap_sdr.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase3_v1_remix/best_audio.pt \
  --model-variant v1 \
  --out-dir /kaggle/working/sdr_phase3_v1_remix_overlap \
  --device cuda \
  --overlap 0.5

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1264.00it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [07:40<00:00,  9.21s/it]
SUMMARY
bass: 3.580 dB
drums: 5.549 dB
other: 3.247 dB
vocal: 5.063 dB
mean: 4.360 dB
saved: /kaggle/working/sdr_phase3_v1_remix_overlap/student_overlap_sdr_results.json


In [35]:
%%writefile /kaggle/working/train_v1_sisdr_weighted.py
import argparse
import json
import random
import sys
from pathlib import Path

import soundfile as sf
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModel

sys.path.insert(0, "/kaggle/working/project")

from student.rope_unet import RopeReplacementUNet
from student.teacher_features import (
    waveform_to_teacher_features,
    teacher_rope_target,
    student_features_to_stems,
)

STEMS = ("bass", "drums", "other", "vocal")
STEM_WEIGHTS = torch.tensor([1.6, 1.0, 1.8, 1.1]).view(1, 4, 1, 1)


def torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def find_audio(song_dir, names):
    for name in names:
        for ext in (".wav", ".flac"):
            p = song_dir / f"{name}{ext}"
            if p.exists():
                return p
    raise FileNotFoundError(f"Missing {names} in {song_dir}")


def find_songs(root):
    root = Path(root)
    return sorted(
        p for p in root.rglob("*")
        if p.is_dir() and ((p / "mixture.wav").exists() or (p / "mixture.flac").exists())
    )


def read_segment(path, start, frames):
    info = sf.info(path)
    start = max(0, min(start, max(0, info.frames - 1)))
    audio, _ = sf.read(path, start=start, frames=frames, dtype="float32", always_2d=True)
    x = torch.from_numpy(audio).transpose(0, 1).contiguous()
    if x.shape[0] == 1:
        x = x.repeat(2, 1)
    elif x.shape[0] > 2:
        x = x[:2]
    if x.shape[-1] < frames:
        x = F.pad(x, (0, frames - x.shape[-1]))
    return x


def stem_paths(song_dir):
    return {
        "bass": find_audio(song_dir, ("bass", "target_bass")),
        "drums": find_audio(song_dir, ("drums", "target_drums")),
        "other": find_audio(song_dir, ("other", "target_other")),
        "vocal": find_audio(song_dir, ("vocals", "vocal", "target_vocals", "target_vocal")),
    }


class Sampler:
    def __init__(self, songs, chunk_size, remix_prob=0.3):
        self.chunk_size = chunk_size
        self.remix_prob = remix_prob
        self.items = []
        for song in songs:
            mix = find_audio(song, ("mixture",))
            info = sf.info(mix)
            self.items.append({"mix": mix, "stems": stem_paths(song), "frames": info.frames})

    def sample_one(self):
        if random.random() < self.remix_prob:
            stems = []
            for stem in STEMS:
                item = random.choice(self.items)
                start = random.randint(0, max(0, item["frames"] - self.chunk_size))
                stems.append(read_segment(item["stems"][stem], start, self.chunk_size))
            stems = torch.stack(stems, dim=0)
            gains = torch.empty(4, 1, 1).uniform_(0.8, 1.2)
            stems = stems * gains
            mix = stems.sum(dim=0).clamp(-1.0, 1.0)
            return mix, stems

        item = random.choice(self.items)
        start = random.randint(0, max(0, item["frames"] - self.chunk_size))
        mix = read_segment(item["mix"], start, self.chunk_size)
        stems = torch.stack([read_segment(item["stems"][s], start, self.chunk_size) for s in STEMS], dim=0)
        gain = random.uniform(0.85, 1.15)
        mix = mix * gain
        stems = stems * gain
        if random.random() < 0.5:
            mix = mix.flip(0)
            stems = stems.flip(1)
        return mix, stems

    def batch(self, batch_size):
        mixes, stems = zip(*(self.sample_one() for _ in range(batch_size)))
        return torch.stack(mixes, 0), torch.stack(stems, 0)


def weighted_l1(pred, target, stem_weights):
    return ((pred - target).abs() * stem_weights.to(pred.device)).mean()


def si_sdr_loss(pred, target, stem_weights, eps=1e-8):
    # pred/target: [B, 4, 2, T]
    pred = pred.float()
    target = target.float()
    pred = pred - pred.mean(dim=-1, keepdim=True)
    target = target - target.mean(dim=-1, keepdim=True)

    dot = (pred * target).sum(dim=-1, keepdim=True)
    target_energy = target.pow(2).sum(dim=-1, keepdim=True) + eps
    proj = dot * target / target_energy
    noise = pred - proj

    ratio = (proj.pow(2).sum(dim=-1) + eps) / (noise.pow(2).sum(dim=-1) + eps)
    sisdr = 10.0 * torch.log10(ratio + eps)  # [B, 4, 2]

    # average stereo channels, then stem-weight
    sisdr = sisdr.mean(dim=-1)  # [B, 4]
    w = stem_weights.view(1, 4).to(pred.device)
    return -(sisdr * w).mean()


def mrstft_loss(pred, target):
    pred = pred.reshape(-1, pred.shape[-1]).float()
    target = target.reshape(-1, target.shape[-1]).float()
    total = pred.new_tensor(0.0)
    for n_fft, hop in ((1024, 256), (2048, 512), (4096, 1024)):
        window = torch.hann_window(n_fft, device=pred.device, dtype=torch.float32)
        p = torch.stft(pred, n_fft=n_fft, hop_length=hop, win_length=n_fft, window=window, return_complex=True)
        t = torch.stft(target, n_fft=n_fft, hop_length=hop, win_length=n_fft, window=window, return_complex=True)
        mag_loss = F.l1_loss(torch.log1p(p.abs()), torch.log1p(t.abs()))
        sc_loss = (p.abs() - t.abs()).norm(p="fro") / (t.abs().norm(p="fro") + 1e-8)
        total = total + mag_loss + 0.1 * sc_loss
    return total / 3.0


def forward_student(teacher, student, mix):
    x = waveform_to_teacher_features(teacher, mix)
    h = student(x)
    stems = student_features_to_stems(teacher, h, mix.shape[-1], mix)
    return x, h, stems


@torch.no_grad()
def validate(teacher, student, sampler, args):
    student.eval()
    vals = []
    stem_weights = STEM_WEIGHTS.to(args.device)
    for _ in tqdm(range(args.val_steps), desc="val"):
        mix, gt = sampler.batch(args.batch_size)
        mix = mix.to(args.device)
        gt = gt.to(args.device)
        _, _, pred = forward_student(teacher, student, mix)
        vals.append(float(weighted_l1(pred, gt, stem_weights).cpu()))
    student.train()
    return sum(vals) / max(1, len(vals))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--project", required=True)
    parser.add_argument("--dataset-root", required=True)
    parser.add_argument("--init-checkpoint", required=True)
    parser.add_argument("--run-dir", default="/kaggle/working/runs/phase5_v1_sisdr_weighted")
    parser.add_argument("--train-songs", type=int, default=90)
    parser.add_argument("--val-songs", type=int, default=10)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--steps-per-epoch", type=int, default=300)
    parser.add_argument("--val-steps", type=int, default=30)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--chunk-seconds", type=float, default=8.0)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--remix-prob", type=float, default=0.3)
    parser.add_argument("--device", default="cuda")
    parser.add_argument("--wave-weight", type=float, default=1.0)
    parser.add_argument("--sisdr-weight", type=float, default=0.02)
    parser.add_argument("--stft-weight", type=float, default=0.2)
    parser.add_argument("--hidden-weight", type=float, default=0.05)
    parser.add_argument("--teacher-stem-weight", type=float, default=0.2)
    parser.add_argument("--mixture-weight", type=float, default=0.05)
    args = parser.parse_args()

    random.seed(1234)
    torch.manual_seed(1234)
    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    teacher = AutoModel.from_pretrained(
        str(Path(args.project) / "teacher_model"),
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(args.device).float().eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    sr = int(teacher.config.wave_sample_rate)
    chunk_size = int(args.chunk_seconds * sr)

    songs = find_songs(Path(args.dataset_root))
    train_songs = songs[:args.train_songs]
    val_songs = songs[args.train_songs:args.train_songs + args.val_songs]

    train_sampler = Sampler(train_songs, chunk_size, remix_prob=args.remix_prob)
    val_sampler = Sampler(val_songs, chunk_size, remix_prob=0.0)

    student = RopeReplacementUNet(hidden_size=int(teacher.config.hidden_size)).to(args.device)
    ckpt = torch_load(args.init_checkpoint, args.device)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    student.load_state_dict(state)
    student.train()

    opt = torch.optim.AdamW(student.parameters(), lr=args.lr, weight_decay=1e-4)
    stem_weights = STEM_WEIGHTS.to(args.device)

    print("student params:", sum(p.numel() for p in student.parameters()))
    print("stem weights:", STEM_WEIGHTS.view(-1).tolist())

    best = float("inf")
    history = []

    for epoch in range(1, args.epochs + 1):
        running = 0.0
        pbar = tqdm(range(args.steps_per_epoch), desc=f"epoch {epoch}")
        for _ in pbar:
            mix, gt = train_sampler.batch(args.batch_size)
            mix = mix.to(args.device)
            gt = gt.to(args.device)

            opt.zero_grad(set_to_none=True)
            x, h, pred = forward_student(teacher, student, mix)

            loss = pred.new_tensor(0.0)
            loss = loss + args.wave_weight * weighted_l1(pred, gt, stem_weights)
            loss = loss + args.sisdr_weight * si_sdr_loss(pred, gt, stem_weights)
            loss = loss + args.stft_weight * mrstft_loss(pred, gt)
            loss = loss + args.mixture_weight * F.l1_loss(pred.sum(dim=1), mix)

            if args.hidden_weight > 0 or args.teacher_stem_weight > 0:
                with torch.inference_mode():
                    th = teacher_rope_target(teacher, x)
                    ts = student_features_to_stems(teacher, th, mix.shape[-1], mix)
                loss = loss + args.hidden_weight * F.l1_loss(h, th)
                loss = loss + args.teacher_stem_weight * weighted_l1(pred, ts, stem_weights)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()

            lv = float(loss.detach().cpu())
            running += lv
            pbar.set_postfix(loss=f"{lv:.4f}")

        train_loss = running / max(1, args.steps_per_epoch)
        val_loss = validate(teacher, student, val_sampler, args)

        print(f"epoch {epoch}: train {train_loss:.6f} | val {val_loss:.6f}")

        ckpt = {
            "model": student.state_dict(),
            "optimizer": opt.state_dict(),
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "model_variant": "v1",
            "hidden_size": int(teacher.config.hidden_size),
            "args": vars(args),
        }

        torch.save(ckpt, run_dir / "last_audio.pt")
        if val_loss < best:
            best = val_loss
            torch.save(ckpt, run_dir / "best_audio.pt")
            print(f"new best: {best:.6f}")

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        (run_dir / "history.json").write_text(json.dumps(history, indent=2))


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/train_v1_sisdr_weighted.py


In [30]:
!python /kaggle/working/train_v1_sisdr_weighted.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --init-checkpoint /kaggle/working/runs/phase3_v1_remix/best_audio.pt \
  --run-dir /kaggle/working/runs/phase5_v1_sisdr_weighted \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 15 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 2e-5 \
  --remix-prob 0.3 \
  --sisdr-weight 0.02 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1247.12it/s, Materializing para
student params: 5762304
stem weights: [1.2999999523162842, 1.0, 1.600000023841858, 1.2000000476837158]
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.03it/s]
epoch 1: train 0.414443 | val 0.031935
new best: 0.031935
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.08it/s]
epoch 2: train 0.382096 | val 0.032656
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.26it/s]
epoch 3: train 0.389652 | val 0.039890
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.87it/s]
epoch 4: train 0.371872 | val 0.029934
new best: 0.029934
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.27it/s]
epoch 5: train 0.386851 | val 0.031995
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.01it/s]
epoch 6: train 0.380604 | val 0.038990
val: 100

In [32]:
!python /kaggle/working/eval_student_overlap_sdr.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase5_v1_sisdr_weighted/best_audio.pt \
  --model-variant v1 \
  --out-dir /kaggle/working/sdr_phase5_v1_sisdr_weighted_overlap \
  --device cuda \
  --overlap 0.5

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1018.07it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [07:21<00:00,  8.83s/it]
SUMMARY
bass: 3.745 dB
drums: 5.764 dB
other: 3.323 dB
vocal: 5.043 dB
mean: 4.469 dB
saved: /kaggle/working/sdr_phase5_v1_sisdr_weighted_overlap/student_overlap_sdr_results.json


In [11]:
!find /kaggle/input -maxdepth 5 -type f | sed 's#^#/##' | grep -E "best_audio|\\.pt|final_all"

//kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload/student_best_audio_weights_only.pt
//kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload/best_audio_training_checkpoint.pt
//kaggle/input/datasets/sujeethkasukurthi/ffffff/best_audio (3).pt


In [10]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*.pt"):
    print(p)

/kaggle/input/datasets/sujeethkasukurthi/ffffff/best_audio (3).pt
/kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload/student_best_audio_weights_only.pt
/kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload/best_audio_training_checkpoint.pt
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/project_code/student_best_audio_weights_only.pt
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/project_code/best_audio_training_checkpoint.pt
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/runs/phase1_v1_audio/last_audio.pt
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/runs/phase1_v1_audio/best_audio.pt
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/runs/runs/phase2_v1_teacher_audio/last_audio.pt
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/runs/runs/phase2_v1_teacher_audi

In [12]:
import torch

BASE_CKPT = "/kaggle/input/datasets/sujeethkasukurthi/ffffff/best_audio (3).pt"
ckpt = torch.load(BASE_CKPT, map_location="cpu", weights_only=False)

print(type(ckpt))
print(ckpt.keys() if isinstance(ckpt, dict) else "raw state dict")
if isinstance(ckpt, dict):
    print("epoch:", ckpt.get("epoch"))
    print("val_loss:", ckpt.get("val_loss"))
    print("model_variant:", ckpt.get("model_variant"))

<class 'dict'>
dict_keys(['model', 'optimizer', 'epoch', 'train_loss', 'val_loss', 'model_variant', 'hidden_size', 'args'])
epoch: 5
val_loss: 0.021829419850837438
model_variant: v1


In [14]:
%%writefile /kaggle/working/eval_student_overlap_sdr.py
import argparse, json, sys
from pathlib import Path
import soundfile as sf
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModel

sys.path.insert(0, "/kaggle/working/project")

from student.rope_unet import RopeReplacementUNet
from student.teacher_features import waveform_to_teacher_features, student_features_to_stems

STEMS = ("bass", "drums", "other", "vocal")

def torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def read_audio(path, sample_rate):
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    x = torch.from_numpy(audio).transpose(0, 1).contiguous()
    if x.shape[0] == 1:
        x = x.repeat(2, 1)
    elif x.shape[0] > 2:
        x = x[:2]
    if sr != sample_rate:
        new_len = round(x.shape[-1] * sample_rate / sr)
        x = F.interpolate(x.unsqueeze(0), size=new_len, mode="linear", align_corners=False).squeeze(0)
    return x

def find_audio(song_dir, names):
    for name in names:
        for ext in (".wav", ".flac"):
            p = song_dir / f"{name}{ext}"
            if p.exists():
                return p
    raise FileNotFoundError(f"Missing {names} in {song_dir}")

def find_songs(root):
    root = Path(root)
    return sorted(p for p in root.rglob("*") if p.is_dir() and ((p/"mixture.wav").exists() or (p/"mixture.flac").exists()))

def simple_sdr(pred, target):
    n = min(pred.shape[-1], target.shape[-1])
    pred = pred[..., :n].float()
    target = target[..., :n].float()
    return float(10 * torch.log10((target.pow(2).sum() + 1e-8) / ((target - pred).pow(2).sum() + 1e-8)))

@torch.no_grad()
def separate_overlap(teacher, student, mixture, chunk_size, hop_size, device):
    total_len = mixture.shape[-1]
    output = torch.zeros(4, 2, total_len)
    weight_sum = torch.zeros(total_len)

    weight = torch.hann_window(chunk_size, periodic=False).float().clamp_min(0.05)
    starts = list(range(0, max(1, total_len - chunk_size + 1), hop_size))
    if not starts or starts[-1] + chunk_size < total_len:
        starts.append(max(0, total_len - chunk_size))

    for start in starts:
        end = min(start + chunk_size, total_len)
        chunk = mixture[:, start:end]
        real_len = chunk.shape[-1]
        if real_len < chunk_size:
            chunk = F.pad(chunk, (0, chunk_size - real_len))

        raw = chunk.unsqueeze(0).to(device)
        x = waveform_to_teacher_features(teacher, raw)
        h = student(x)
        pred = student_features_to_stems(teacher, h, chunk_size, raw).cpu()[0, :, :, :real_len]

        w = weight[:real_len]
        output[:, :, start:end] += pred * w.view(1, 1, -1)
        weight_sum[start:end] += w

    return output / weight_sum.clamp_min(1e-8).view(1, 1, -1)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--project", required=True)
    ap.add_argument("--dataset-root", required=True)
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--out-dir", default="/kaggle/working/sdr_overlap")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--overlap", type=float, default=0.5)
    ap.add_argument("--chunk-seconds", type=float, default=8.0)
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    teacher = AutoModel.from_pretrained(
        str(Path(args.project) / "teacher_model"),
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(args.device).float().eval()

    for p in teacher.parameters():
        p.requires_grad_(False)

    ckpt = torch_load(args.checkpoint, args.device)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    student = RopeReplacementUNet(hidden_size=int(teacher.config.hidden_size)).to(args.device).eval()
    student.load_state_dict(state)

    sr = int(teacher.config.wave_sample_rate)
    chunk_size = int(args.chunk_seconds * sr)
    hop_size = int(chunk_size * (1 - args.overlap))

    songs = find_songs(args.dataset_root)
    totals = {s: [] for s in STEMS}
    rows = []

    for song_dir in tqdm(songs, desc="songs"):
        mix = read_audio(find_audio(song_dir, ("mixture",)), sr)
        targets = {
            "bass": read_audio(find_audio(song_dir, ("bass", "target_bass")), sr),
            "drums": read_audio(find_audio(song_dir, ("drums", "target_drums")), sr),
            "other": read_audio(find_audio(song_dir, ("other", "target_other")), sr),
            "vocal": read_audio(find_audio(song_dir, ("vocals", "vocal", "target_vocals", "target_vocal")), sr),
        }

        pred = separate_overlap(teacher, student, mix, chunk_size, hop_size, args.device)

        row = {"song": song_dir.name}
        for i, stem in enumerate(STEMS):
            row[stem] = simple_sdr(pred[i], targets[stem])
            totals[stem].append(row[stem])
        row["mean"] = sum(row[s] for s in STEMS) / 4
        rows.append(row)
        print(f"{song_dir.name}: bass {row['bass']:.3f} | drums {row['drums']:.3f} | other {row['other']:.3f} | vocal {row['vocal']:.3f} | mean {row['mean']:.3f}")

    summary = {stem: sum(vals)/len(vals) for stem, vals in totals.items()}
    summary["mean"] = sum(summary.values()) / 4
    print("SUMMARY")
    for stem in STEMS:
        print(f"{stem}: {summary[stem]:.3f} dB")
    print(f"mean: {summary['mean']:.3f} dB")

    out_path = out_dir / "student_overlap_sdr_results.json"
    out_path.write_text(json.dumps({"summary": summary, "songs": rows}, indent=2))
    print(f"saved: {out_path}")

if __name__ == "__main__":
    main()

Writing /kaggle/working/eval_student_overlap_sdr.py


In [16]:
from pathlib import Path

# Find uploaded project folder
for p in Path("/kaggle/input").rglob("kaggle_rope_unet_phases_upload"):
    print(p)

/kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/project_code/kaggle_rope_unet_phases_upload
/kaggle/input/datasets/sujeethkasukurthi/ffffff/final_all_results/final_save/project_code/project/kaggle_rope_unet_phases_upload


In [17]:
from pathlib import Path

src = next(Path("/kaggle/input").rglob("kaggle_rope_unet_phases_upload"))
print("copying from:", src)

!rm -rf /kaggle/working/project
!cp -r "{src}" /kaggle/working/project

PROJECT = "/kaggle/working/project"
MUSDB = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"

!ls -lh "$PROJECT/student"

copying from: /kaggle/input/datasets/sujeethkasukurthi/abcdef/kaggle_rope_unet_phases_upload
total 36K
-rw-r--r-- 1 root root 4.2K May 31 13:34 axial_rope_unet_v2.py
-rw-r--r-- 1 root root 2.6K May 31 13:34 baby_separator.py
-rw-r--r-- 1 root root   29 May 31 13:34 __init__.py
-rw-r--r-- 1 root root 3.7K May 31 13:34 rope_unet.py
-rw-r--r-- 1 root root 5.0K May 31 13:34 teacher_features.py
-rw-r--r-- 1 root root 4.1K May 31 13:34 train_rope_unet.py


In [19]:
!python /kaggle/working/eval_student_overlap_sdr.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint "$BASE_CKPT" \
  --out-dir /kaggle/working/sdr_uploaded_base_overlap \
  --device cuda \
  --overlap 0.5

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1305.80it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [09:07<00:00, 10.95s/it]
SUMMARY
bass: 3.580 dB
drums: 5.549 dB
other: 3.247 dB
vocal: 5.063 dB
mean: 4.360 dB
saved: /kaggle/working/sdr_uploaded_base_overlap/student_overlap_sdr_results.json


In [22]:
!python /kaggle/working/train_v1_sisdr_weighted.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --init-checkpoint "$BASE_CKPT" \
  --run-dir /kaggle/working/runs/phase5_v1_sisdr_weighted_recovered \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 10 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 2e-5 \
  --remix-prob 0.3 \
  --sisdr-weight 0.02 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1333.06it/s, Materializing para
student params: 5762304
stem weights: [2.0, 1.0, 2.0, 1.5]
val: 100%|██████████████████████████████████████| 30/30 [00:07<00:00,  4.22it/s]
epoch 1: train 0.428109 | val 0.041472
new best: 0.041472
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.32it/s]
epoch 2: train 0.382975 | val 0.042194
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.54it/s]
epoch 3: train 0.389725 | val 0.051658
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.30it/s]
epoch 4: train 0.368601 | val 0.038636
new best: 0.038636
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.22it/s]
epoch 5: train 0.388706 | val 0.041025
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.86it/s]
epoch 6: train 0.376582 | val 0.050414
val: 100%|██████████████████████████████████████| 30

In [23]:
!python /kaggle/working/eval_student_overlap_sdr.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase5_v1_sisdr_weighted_recovered/best_audio.pt \
  --out-dir /kaggle/working/sdr_phase5_recovered_best_overlap \
  --device cuda \
  --overlap 0.5

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1295.63it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [07:13<00:00,  8.68s/it]
SUMMARY
bass: 3.737 dB
drums: 5.764 dB
other: 3.325 dB
vocal: 5.043 dB
mean: 4.468 dB
saved: /kaggle/working/sdr_phase5_recovered_best_overlap/student_overlap_sdr_results.json


In [24]:
!mkdir -p /kaggle/working/final_save_recovered

# Save model/checkpoints
!cp -r /kaggle/working/runs /kaggle/working/final_save_recovered/runs 2>/dev/null || true

# Save all SDR folders
!cp -r /kaggle/working/sdr_uploaded_base_overlap /kaggle/working/final_save_recovered/sdr_uploaded_base_overlap 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase5_recovered_best_overlap /kaggle/working/final_save_recovered/sdr_phase5_recovered_best_overlap 2>/dev/null || true
!cp -r /kaggle/working/sdr_phase5_recovered_last_overlap /kaggle/working/final_save_recovered/sdr_phase5_recovered_last_overlap 2>/dev/null || true

# Save project code
!cp -r "$PROJECT" /kaggle/working/final_save_recovered/project_code 2>/dev/null || true

# Save selected best checkpoint/result separately
!mkdir -p /kaggle/working/final_save_recovered/final_selected_model
!cp /kaggle/working/runs/phase5_v1_sisdr_weighted_recovered/best_audio.pt /kaggle/working/final_save_recovered/final_selected_model/best_phase5_recovered.pt
!cp /kaggle/working/sdr_phase5_recovered_best_overlap/student_overlap_sdr_results.json /kaggle/working/final_save_recovered/final_selected_model/best_phase5_recovered_sdr.json

# Zip
!cd /kaggle/working && zip -r final_save_recovered.zip final_save_recovered

  adding: final_save_recovered/ (stored 0%)
  adding: final_save_recovered/final_selected_model/ (stored 0%)
  adding: final_save_recovered/final_selected_model/best_phase5_recovered_sdr.json (deflated 64%)
  adding: final_save_recovered/final_selected_model/best_phase5_recovered.pt (deflated 8%)
  adding: final_save_recovered/runs/ (stored 0%)
  adding: final_save_recovered/runs/phase5_v1_sisdr_weighted_recovered/ (stored 0%)
  adding: final_save_recovered/runs/phase5_v1_sisdr_weighted_recovered/last_audio.pt (deflated 8%)
  adding: final_save_recovered/runs/phase5_v1_sisdr_weighted_recovered/history.json (deflated 71%)
  adding: final_save_recovered/runs/phase5_v1_sisdr_weighted_recovered/best_audio.pt (deflated 8%)
  adding: final_save_recovered/sdr_phase5_recovered_best_overlap/ (stored 0%)
  adding: final_save_recovered/sdr_phase5_recovered_best_overlap/student_overlap_sdr_results.json (deflated 64%)
  adding: final_save_recovered/project_code/ (stored 0%)
  adding: final_save_rec

In [25]:
import json
from pathlib import Path

for p in Path("/kaggle/working").rglob("*student_overlap_sdr_results.json"):
    data = json.loads(p.read_text())
    print("\nFILE:", p)
    for k, v in data["summary"].items():
        print(f"{k}: {v:.3f}")


FILE: /kaggle/working/sdr_phase5_recovered_best_overlap/student_overlap_sdr_results.json
bass: 3.737
drums: 5.764
other: 3.325
vocal: 5.043
mean: 4.468

FILE: /kaggle/working/sdr_uploaded_base_overlap/student_overlap_sdr_results.json
bass: 3.580
drums: 5.549
other: 3.247
vocal: 5.063
mean: 4.360

FILE: /kaggle/working/final_save_recovered/sdr_phase5_recovered_best_overlap/student_overlap_sdr_results.json
bass: 3.737
drums: 5.764
other: 3.325
vocal: 5.043
mean: 4.468

FILE: /kaggle/working/final_save_recovered/sdr_uploaded_base_overlap/student_overlap_sdr_results.json
bass: 3.580
drums: 5.549
other: 3.247
vocal: 5.063
mean: 4.360


In [28]:
%%writefile /kaggle/working/eval_student_overlap_mc_sdr.py
import argparse, json, sys
from pathlib import Path
import soundfile as sf
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModel

sys.path.insert(0, "/kaggle/working/project")

from student.rope_unet import RopeReplacementUNet
from student.teacher_features import waveform_to_teacher_features, student_features_to_stems

STEMS = ("bass", "drums", "other", "vocal")

def torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def read_audio(path, sample_rate):
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    x = torch.from_numpy(audio).transpose(0, 1).contiguous()
    if x.shape[0] == 1:
        x = x.repeat(2, 1)
    elif x.shape[0] > 2:
        x = x[:2]
    if sr != sample_rate:
        new_len = round(x.shape[-1] * sample_rate / sr)
        x = F.interpolate(x.unsqueeze(0), size=new_len, mode="linear", align_corners=False).squeeze(0)
    return x

def find_audio(song_dir, names):
    for name in names:
        for ext in (".wav", ".flac"):
            p = song_dir / f"{name}{ext}"
            if p.exists():
                return p
    raise FileNotFoundError(f"Missing {names} in {song_dir}")

def find_songs(root):
    root = Path(root)
    return sorted(p for p in root.rglob("*") if p.is_dir() and ((p/"mixture.wav").exists() or (p/"mixture.flac").exists()))

def simple_sdr(pred, target):
    n = min(pred.shape[-1], target.shape[-1])
    pred = pred[..., :n].float()
    target = target[..., :n].float()
    return float(10 * torch.log10((target.pow(2).sum() + 1e-8) / ((target - pred).pow(2).sum() + 1e-8)))

def mixture_consistency(pred, mix, strength=1.0):
    # pred: [4, 2, T], mix: [2, T]
    residual = mix - pred.sum(dim=0)
    return pred + strength * residual.unsqueeze(0) / pred.shape[0]

@torch.no_grad()
def separate_overlap(teacher, student, mixture, chunk_size, hop_size, device):
    total_len = mixture.shape[-1]
    output = torch.zeros(4, 2, total_len)
    weight_sum = torch.zeros(total_len)

    weight = torch.hann_window(chunk_size, periodic=False).float().clamp_min(0.05)
    starts = list(range(0, max(1, total_len - chunk_size + 1), hop_size))
    if not starts or starts[-1] + chunk_size < total_len:
        starts.append(max(0, total_len - chunk_size))

    for start in starts:
        end = min(start + chunk_size, total_len)
        chunk = mixture[:, start:end]
        real_len = chunk.shape[-1]
        if real_len < chunk_size:
            chunk = F.pad(chunk, (0, chunk_size - real_len))

        raw = chunk.unsqueeze(0).to(device)
        x = waveform_to_teacher_features(teacher, raw)
        h = student(x)
        pred = student_features_to_stems(teacher, h, chunk_size, raw).cpu()[0, :, :, :real_len]

        w = weight[:real_len]
        output[:, :, start:end] += pred * w.view(1, 1, -1)
        weight_sum[start:end] += w

    return output / weight_sum.clamp_min(1e-8).view(1, 1, -1)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--project", required=True)
    ap.add_argument("--dataset-root", required=True)
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--out-dir", default="/kaggle/working/sdr_mc")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--overlap", type=float, default=0.5)
    ap.add_argument("--mc-strength", type=float, default=1.0)
    ap.add_argument("--chunk-seconds", type=float, default=8.0)
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    teacher = AutoModel.from_pretrained(
        str(Path(args.project) / "teacher_model"),
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(args.device).float().eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    ckpt = torch_load(args.checkpoint, args.device)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    student = RopeReplacementUNet(hidden_size=int(teacher.config.hidden_size)).to(args.device).eval()
    student.load_state_dict(state)

    sr = int(teacher.config.wave_sample_rate)
    chunk_size = int(args.chunk_seconds * sr)
    hop_size = int(chunk_size * (1 - args.overlap))

    totals = {s: [] for s in STEMS}
    rows = []

    for song_dir in tqdm(find_songs(args.dataset_root), desc="songs"):
        mix = read_audio(find_audio(song_dir, ("mixture",)), sr)
        targets = {
            "bass": read_audio(find_audio(song_dir, ("bass", "target_bass")), sr),
            "drums": read_audio(find_audio(song_dir, ("drums", "target_drums")), sr),
            "other": read_audio(find_audio(song_dir, ("other", "target_other")), sr),
            "vocal": read_audio(find_audio(song_dir, ("vocals", "vocal", "target_vocals", "target_vocal")), sr),
        }

        pred = separate_overlap(teacher, student, mix, chunk_size, hop_size, args.device)
        pred = mixture_consistency(pred, mix, strength=args.mc_strength)

        row = {"song": song_dir.name}
        for i, stem in enumerate(STEMS):
            row[stem] = simple_sdr(pred[i], targets[stem])
            totals[stem].append(row[stem])
        row["mean"] = sum(row[s] for s in STEMS) / 4
        rows.append(row)

        print(f"{song_dir.name}: bass {row['bass']:.3f} | drums {row['drums']:.3f} | other {row['other']:.3f} | vocal {row['vocal']:.3f} | mean {row['mean']:.3f}")

    summary = {stem: sum(vals)/len(vals) for stem, vals in totals.items()}
    summary["mean"] = sum(summary.values()) / 4

    print("SUMMARY")
    for stem in STEMS:
        print(f"{stem}: {summary[stem]:.3f} dB")
    print(f"mean: {summary['mean']:.3f} dB")

    out_path = out_dir / "student_overlap_mc_sdr_results.json"
    out_path.write_text(json.dumps({"summary": summary, "songs": rows}, indent=2))
    print(f"saved: {out_path}")

if __name__ == "__main__":
    main()

Overwriting /kaggle/working/eval_student_overlap_mc_sdr.py


In [29]:
BEST_CKPT = "/kaggle/working/runs/phase5_v1_sisdr_weighted_recovered/best_audio.pt"

!python /kaggle/working/eval_student_overlap_mc_sdr.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint "$BEST_CKPT" \
  --out-dir /kaggle/working/sdr_phase5_mc_overlap \
  --device cuda \
  --overlap 0.5 \
  --mc-strength 1.0

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1251.19it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [07:11<00:00,  8.63s/it]
SUMMARY
bass: 3.741 dB
drums: 5.764 dB
other: 3.307 dB
vocal: 5.062 dB
mean: 4.469 dB
saved: /kaggle/working/sdr_phase5_mc_overlap/student_overlap_mc_sdr_results.json


In [31]:
# Edit existing SI-SDR script stem weights in memory by writing a new milder script
from pathlib import Path

src = Path("/kaggle/working/train_v1_sisdr_weighted.py")
dst = Path("/kaggle/working/train_v1_sisdr_mild_focus.py")

text = src.read_text()
text = text.replace(
    "STEM_WEIGHTS = torch.tensor([1.3, 1.0, 1.6, 1.2]).view(1, 4, 1, 1)",
    "STEM_WEIGHTS = torch.tensor([1.6, 1.0, 1.8, 1.1]).view(1, 4, 1, 1)"
)
dst.write_text(text)
print("wrote", dst)

wrote /kaggle/working/train_v1_sisdr_mild_focus.py


In [36]:
!python /kaggle/working/train_v1_sisdr_mild_focus.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --init-checkpoint /kaggle/working/runs/phase5_v1_sisdr_weighted_recovered/best_audio.pt \
  --run-dir /kaggle/working/runs/phase7_v1_mild_focus \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 5 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 1e-5 \
  --remix-prob 0.2 \
  --sisdr-weight 0.03 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1300.87it/s, Materializing para
student params: 5762304
stem weights: [2.0, 1.0, 2.0, 1.5]
epoch 1:  53%|██████████▏        | 160/300 [01:52<01:38,  1.43it/s, loss=0.3386]
Traceback (most recent call last):
  File "/kaggle/working/train_v1_sisdr_mild_focus.py", line 295, in <module>
    main()
  File "/kaggle/working/train_v1_sisdr_mild_focus.py", line 260, in main
    loss.backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/__init__.py", line 364, in backward
    _engine_run_backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py", line 865, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [33]:
!python /kaggle/working/eval_student_overlap_mc_sdr.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase7_v1_mild_focus/best_audio.pt \
  --out-dir /kaggle/working/sdr_phase7_mild_focus_best_mc \
  --device cuda \
  --overlap 0.5 \
  --mc-strength 1.0

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1287.42it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [07:16<00:00,  8.73s/it]
SUMMARY
bass: 3.704 dB
drums: 5.760 dB
other: 3.329 dB
vocal: 5.084 dB
mean: 4.469 dB
saved: /kaggle/working/sdr_phase7_mild_focus_best_mc/student_overlap_mc_sdr_results.json


In [37]:
!mkdir -p /kaggle/working/final_selected_phase5_mc
!cp /kaggle/working/runs/phase5_v1_sisdr_weighted_recovered/best_audio.pt /kaggle/working/final_selected_phase5_mc/best_audio_phase5_mc.pt
!cp /kaggle/working/sdr_phase5_mc_overlap/student_overlap_mc_sdr_results.json /kaggle/working/final_selected_phase5_mc/sdr_phase5_mc.json
!cp /kaggle/working/eval_student_overlap_mc_sdr.py /kaggle/working/final_selected_phase5_mc/eval_student_overlap_mc_sdr.py
!cp /kaggle/working/train_v1_sisdr_weighted.py /kaggle/working/final_selected_phase5_mc/train_v1_sisdr_weighted.py

!cd /kaggle/working && zip -r final_selected_phase5_mc.zip final_selected_phase5_mc

  adding: final_selected_phase5_mc/ (stored 0%)
  adding: final_selected_phase5_mc/eval_student_overlap_mc_sdr.py (deflated 63%)
  adding: final_selected_phase5_mc/sdr_phase5_mc.json (deflated 64%)
  adding: final_selected_phase5_mc/best_audio_phase5_mc.pt (deflated 8%)
  adding: final_selected_phase5_mc/train_v1_sisdr_weighted.py (deflated 69%)


In [39]:
%%writefile /kaggle/working/train_v1_mask_distill.py
import argparse, json, random, sys
from pathlib import Path

import soundfile as sf
import torch
import torch.nn.functional as F
from einops import rearrange
from tqdm import tqdm
from transformers import AutoModel

sys.path.insert(0, "/kaggle/working/project")

from student.rope_unet import RopeReplacementUNet
from student.teacher_features import waveform_to_teacher_features, teacher_rope_target, student_features_to_stems

STEMS = ("bass", "drums", "other", "vocal")
STEM_WEIGHTS = torch.tensor([1.5, 1.0, 1.7, 1.1]).view(1, 4, 1, 1)


def torch_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def find_audio(song_dir, names):
    for name in names:
        for ext in (".wav", ".flac"):
            p = song_dir / f"{name}{ext}"
            if p.exists():
                return p
    raise FileNotFoundError(f"Missing {names} in {song_dir}")


def find_songs(root):
    root = Path(root)
    return sorted(
        p for p in root.rglob("*")
        if p.is_dir() and ((p / "mixture.wav").exists() or (p / "mixture.flac").exists())
    )


def read_segment(path, start, frames):
    info = sf.info(path)
    start = max(0, min(start, max(0, info.frames - 1)))
    audio, _ = sf.read(path, start=start, frames=frames, dtype="float32", always_2d=True)
    x = torch.from_numpy(audio).transpose(0, 1).contiguous()
    if x.shape[0] == 1:
        x = x.repeat(2, 1)
    elif x.shape[0] > 2:
        x = x[:2]
    if x.shape[-1] < frames:
        x = F.pad(x, (0, frames - x.shape[-1]))
    return x


def stem_paths(song_dir):
    return {
        "bass": find_audio(song_dir, ("bass", "target_bass")),
        "drums": find_audio(song_dir, ("drums", "target_drums")),
        "other": find_audio(song_dir, ("other", "target_other")),
        "vocal": find_audio(song_dir, ("vocals", "vocal", "target_vocals", "target_vocal")),
    }


class Sampler:
    def __init__(self, songs, chunk_size, remix_prob=0.2):
        self.chunk_size = chunk_size
        self.remix_prob = remix_prob
        self.items = []
        for song in songs:
            mix = find_audio(song, ("mixture",))
            info = sf.info(mix)
            self.items.append({"mix": mix, "stems": stem_paths(song), "frames": info.frames})

    def sample_one(self):
        if random.random() < self.remix_prob:
            stems = []
            for stem in STEMS:
                item = random.choice(self.items)
                start = random.randint(0, max(0, item["frames"] - self.chunk_size))
                stems.append(read_segment(item["stems"][stem], start, self.chunk_size))
            stems = torch.stack(stems, 0)
            gains = torch.empty(4, 1, 1).uniform_(0.85, 1.15)
            stems = stems * gains
            mix = stems.sum(0).clamp(-1, 1)
            return mix, stems

        item = random.choice(self.items)
        start = random.randint(0, max(0, item["frames"] - self.chunk_size))
        mix = read_segment(item["mix"], start, self.chunk_size)
        stems = torch.stack([read_segment(item["stems"][s], start, self.chunk_size) for s in STEMS], 0)
        gain = random.uniform(0.9, 1.1)
        mix = mix * gain
        stems = stems * gain
        if random.random() < 0.5:
            mix = mix.flip(0)
            stems = stems.flip(1)
        return mix, stems

    def batch(self, batch_size):
        mixes, stems = zip(*(self.sample_one() for _ in range(batch_size)))
        return torch.stack(mixes, 0), torch.stack(stems, 0)


def weighted_l1(pred, target, weights):
    return ((pred - target).abs() * weights.to(pred.device)).mean()


def si_sdr_loss(pred, target, weights, eps=1e-8):
    pred = pred.float() - pred.float().mean(dim=-1, keepdim=True)
    target = target.float() - target.float().mean(dim=-1, keepdim=True)
    proj = (pred * target).sum(dim=-1, keepdim=True) * target / (target.pow(2).sum(dim=-1, keepdim=True) + eps)
    noise = pred - proj
    ratio = (proj.pow(2).sum(dim=-1) + eps) / (noise.pow(2).sum(dim=-1) + eps)
    sisdr = 10 * torch.log10(ratio + eps)
    sisdr = sisdr.mean(dim=-1)
    w = weights.view(1, 4).to(pred.device)
    return -(sisdr * w).mean()


def mrstft_loss(pred, target):
    pred = pred.reshape(-1, pred.shape[-1]).float()
    target = target.reshape(-1, target.shape[-1]).float()
    total = pred.new_tensor(0.0)
    for n_fft, hop in ((1024, 256), (2048, 512), (4096, 1024)):
        window = torch.hann_window(n_fft, device=pred.device, dtype=torch.float32)
        p = torch.stft(pred, n_fft=n_fft, hop_length=hop, win_length=n_fft, window=window, return_complex=True)
        t = torch.stft(target, n_fft=n_fft, hop_length=hop, win_length=n_fft, window=window, return_complex=True)
        total = total + F.l1_loss(torch.log1p(p.abs()), torch.log1p(t.abs()))
    return total / 3.0


def decode_mask_and_stems(model, hidden, raw_audio):
    """
    Returns:
      mask: [B, 4, 2, F, T, 2]
      stems: [B, 4, 2, samples]
    """
    freq_model = model.freq_domain_model
    b, c, length = raw_audio.shape

    h = freq_model.final_norm(hidden)
    if freq_model.time_conv_length is not None:
        h = freq_model.time_deconv(h)
        h = rearrange(h, "b t n (d tc) -> b (t tc) n d", tc=freq_model.time_conv_length)

    with torch.autocast(device_type=raw_audio.device.type, enabled=False):
        packed = rearrange(raw_audio.float(), "b c t -> (b c) t")
        stft = torch.stft(
            packed,
            **model.stft_out_kwargs,
            window=torch.hann_window(
                model.stft_out_kwargs.get("win_length") or model.stft_out_kwargs["n_fft"],
                periodic=True,
                device=raw_audio.device,
                dtype=torch.float32,
            ),
            return_complex=True,
        )
        stft_real = torch.view_as_real(stft)

    t_frames = stft_real.shape[-2]
    h = h[:, :t_frames, :, :]

    mask = torch.stack([fn(h) for fn in freq_model.mask_estimators], dim=1)
    mask = rearrange(mask, "b n t (f c z) -> b n c f t z", z=2, c=c).float()

    with torch.autocast(device_type=raw_audio.device.type, enabled=False):
        stft_expanded = rearrange(stft_real, "(b c) f t z -> b 1 c f t z", b=b, c=c)
        masked = torch.view_as_complex(stft_expanded) * torch.view_as_complex(mask)
        masked = rearrange(masked, "b n c f t -> (b n c) f t")
        audio = torch.istft(
            masked,
            **model.stft_out_kwargs,
            window=torch.hann_window(
                model.stft_out_kwargs.get("win_length") or model.stft_out_kwargs["n_fft"],
                periodic=True,
                device=raw_audio.device,
                dtype=torch.float32,
            ),
            return_complex=False,
            length=length,
        )
        audio = rearrange(audio, "(b n c) t -> b n c t", b=b, n=model.config.num_stems, c=c)

    return mask, audio


@torch.no_grad()
def validate(teacher, student, sampler, args):
    student.eval()
    vals = []
    weights = STEM_WEIGHTS.to(args.device)
    for _ in tqdm(range(args.val_steps), desc="val"):
        mix, gt = sampler.batch(args.batch_size)
        mix = mix.to(args.device)
        gt = gt.to(args.device)
        x = waveform_to_teacher_features(teacher, mix)
        h = student(x)
        _, pred = decode_mask_and_stems(teacher, h, mix)
        vals.append(float(weighted_l1(pred, gt, weights).cpu()))
    student.train()
    return sum(vals) / max(1, len(vals))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--project", required=True)
    ap.add_argument("--dataset-root", required=True)
    ap.add_argument("--init-checkpoint", required=True)
    ap.add_argument("--run-dir", default="/kaggle/working/runs/phase8_v1_mask_distill")
    ap.add_argument("--train-songs", type=int, default=90)
    ap.add_argument("--val-songs", type=int, default=10)
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--steps-per-epoch", type=int, default=300)
    ap.add_argument("--val-steps", type=int, default=30)
    ap.add_argument("--batch-size", type=int, default=1)
    ap.add_argument("--chunk-seconds", type=float, default=8.0)
    ap.add_argument("--lr", type=float, default=1e-5)
    ap.add_argument("--remix-prob", type=float, default=0.2)
    ap.add_argument("--wave-weight", type=float, default=1.0)
    ap.add_argument("--stft-weight", type=float, default=0.2)
    ap.add_argument("--sisdr-weight", type=float, default=0.03)
    ap.add_argument("--hidden-weight", type=float, default=0.05)
    ap.add_argument("--teacher-stem-weight", type=float, default=0.2)
    ap.add_argument("--mask-weight", type=float, default=0.2)
    ap.add_argument("--mixture-weight", type=float, default=0.05)
    ap.add_argument("--device", default="cuda")
    args = ap.parse_args()

    random.seed(4321)
    torch.manual_seed(4321)

    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    teacher = AutoModel.from_pretrained(
        str(Path(args.project) / "teacher_model"),
        trust_remote_code=True,
        torch_dtype=torch.float32,
    ).to(args.device).float().eval()

    for p in teacher.parameters():
        p.requires_grad_(False)

    songs = find_songs(Path(args.dataset_root))
    train_songs = songs[:args.train_songs]
    val_songs = songs[args.train_songs:args.train_songs + args.val_songs]
    chunk_size = int(args.chunk_seconds * int(teacher.config.wave_sample_rate))

    train_sampler = Sampler(train_songs, chunk_size, remix_prob=args.remix_prob)
    val_sampler = Sampler(val_songs, chunk_size, remix_prob=0.0)

    student = RopeReplacementUNet(hidden_size=int(teacher.config.hidden_size)).to(args.device)
    ckpt = torch_load(args.init_checkpoint, args.device)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    student.load_state_dict(state)
    student.train()

    opt = torch.optim.AdamW(student.parameters(), lr=args.lr, weight_decay=1e-4)
    weights = STEM_WEIGHTS.to(args.device)

    print("student params:", sum(p.numel() for p in student.parameters()))
    print("stem weights:", STEM_WEIGHTS.view(-1).tolist())

    best = float("inf")
    history = []

    for epoch in range(1, args.epochs + 1):
        running = 0.0
        pbar = tqdm(range(args.steps_per_epoch), desc=f"epoch {epoch}")

        for _ in pbar:
            mix, gt = train_sampler.batch(args.batch_size)
            mix = mix.to(args.device)
            gt = gt.to(args.device)

            opt.zero_grad(set_to_none=True)

            x = waveform_to_teacher_features(teacher, mix)
            h_student = student(x)
            mask_student, pred = decode_mask_and_stems(teacher, h_student, mix)

            loss = pred.new_tensor(0.0)
            loss = loss + args.wave_weight * weighted_l1(pred, gt, weights)
            loss = loss + args.stft_weight * mrstft_loss(pred, gt)
            loss = loss + args.sisdr_weight * si_sdr_loss(pred, gt, weights)
            loss = loss + args.mixture_weight * F.l1_loss(pred.sum(dim=1), mix)

            with torch.inference_mode():
                h_teacher = teacher_rope_target(teacher, x)
                mask_teacher, teacher_stems = decode_mask_and_stems(teacher, h_teacher, mix)

            loss = loss + args.hidden_weight * F.l1_loss(h_student, h_teacher)
            loss = loss + args.teacher_stem_weight * weighted_l1(pred, teacher_stems, weights)
            loss = loss + args.mask_weight * F.l1_loss(mask_student, mask_teacher)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()

            lv = float(loss.detach().cpu())
            running += lv
            pbar.set_postfix(loss=f"{lv:.4f}")

        train_loss = running / max(1, args.steps_per_epoch)
        val_loss = validate(teacher, student, val_sampler, args)

        print(f"epoch {epoch}: train {train_loss:.6f} | val {val_loss:.6f}")

        ckpt = {
            "model": student.state_dict(),
            "optimizer": opt.state_dict(),
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "model_variant": "v1",
            "hidden_size": int(teacher.config.hidden_size),
            "args": vars(args),
        }
        torch.save(ckpt, run_dir / "last_audio.pt")
        if val_loss < best:
            best = val_loss
            torch.save(ckpt, run_dir / "best_audio.pt")
            print(f"new best: {best:.6f}")

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        (run_dir / "history.json").write_text(json.dumps(history, indent=2))

if __name__ == "__main__":
    main()

Overwriting /kaggle/working/train_v1_mask_distill.py


In [40]:
!python /kaggle/working/train_v1_mask_distill.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/train" \
  --init-checkpoint /kaggle/working/runs/phase5_v1_sisdr_weighted_recovered/best_audio.pt \
  --run-dir /kaggle/working/runs/phase8_v1_mask_distill \
  --train-songs 90 \
  --val-songs 10 \
  --epochs 6 \
  --steps-per-epoch 300 \
  --val-steps 30 \
  --batch-size 1 \
  --lr 1e-5 \
  --remix-prob 0.2 \
  --mask-weight 0.2 \
  --sisdr-weight 0.03 \
  --device cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1237.27it/s, Materializing para
student params: 5762304
stem weights: [1.5, 1.0, 1.7000000476837158, 1.100000023841858]
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.69it/s]
epoch 1: train 0.359236 | val 0.030000
new best: 0.030000
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.88it/s]
epoch 2: train 0.342653 | val 0.032939
val: 100%|██████████████████████████████████████| 30/30 [00:06<00:00,  4.95it/s]
epoch 3: train 0.351183 | val 0.036026
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.28it/s]
epoch 4: train 0.364458 | val 0.033440
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.65it/s]
epoch 5: train 0.344141 | val 0.037781
val: 100%|██████████████████████████████████████| 30/30 [00:05<00:00,  5.44it/s]
epoch 6: train 0.325171 | val 0.034919


In [41]:
!python /kaggle/working/eval_student_overlap_mc_sdr.py \
  --project "$PROJECT" \
  --dataset-root "$MUSDB/test" \
  --checkpoint /kaggle/working/runs/phase8_v1_mask_distill/best_audio.pt \
  --out-dir /kaggle/working/sdr_phase8_mask_distill_best_mc \
  --device cuda \
  --overlap 0.5 \
  --mc-strength 1.0

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 1771/1771 [00:01<00:00, 1275.13it/s, Materializing para
songs: 100%|████████████████████████████████████| 50/50 [07:15<00:00,  8.71s/it]
SUMMARY
bass: 3.760 dB
drums: 5.778 dB
other: 3.314 dB
vocal: 5.055 dB
mean: 4.477 dB
saved: /kaggle/working/sdr_phase8_mask_distill_best_mc/student_overlap_mc_sdr_results.json


In [42]:
!mkdir -p /kaggle/working/final_selected_best

!cp /kaggle/working/runs/phase8_v1_mask_distill/best_audio.pt /kaggle/working/final_selected_best/best_audio_phase8_mask_distill.pt
!cp /kaggle/working/sdr_phase8_mask_distill_best_mc/student_overlap_mc_sdr_results.json /kaggle/working/final_selected_best/sdr_phase8_mask_distill_best_mc.json

!cp /kaggle/working/train_v1_mask_distill.py /kaggle/working/final_selected_best/train_v1_mask_distill.py 2>/dev/null || true
!cp /kaggle/working/eval_student_overlap_mc_sdr.py /kaggle/working/final_selected_best/eval_student_overlap_mc_sdr.py 2>/dev/null || true

!cd /kaggle/working && zip -r final_selected_best_phase8.zip final_selected_best

  adding: final_selected_best/ (stored 0%)
  adding: final_selected_best/eval_student_overlap_mc_sdr.py (deflated 63%)
  adding: final_selected_best/train_v1_mask_distill.py (deflated 70%)
  adding: final_selected_best/best_audio_phase8_mask_distill.pt (deflated 8%)
  adding: final_selected_best/sdr_phase8_mask_distill_best_mc.json (deflated 64%)
